In [1]:
from pyspark.sql import SparkSession
from delta import *

# Configurando o Spark com suporte nativo ao Delta Lake
builder = SparkSession.builder.appName("DeltaLakeFootball") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")


# Iniciando a sessão configurada
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Sucesso! A sessão Spark com Delta Lake está ativa.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 23:30:52 WARN Utils: Your hostname, DESKTOP-4KKPSDT, resolves to a loopback address: 127.0.1.1; using 172.21.148.61 instead (on interface eth0)
26/05/04 23:30:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/arthurlg81/trabalho-spark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/arthurlg81/.ivy2.5.2/cache
The jars for the packages stored in: /home/arthurlg81/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e1cd4aca-3692-4dae-bc18-25417040cccd;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 

Sucesso! A sessão Spark com Delta Lake está ativa.


In [2]:
# 1. Definição do caminho onde os dados serão salvos (camada Silver)
path_delta = "../data/delta/fato_evento"

# 2. Criando dados iniciais de uma partida (Simulação: Flamengo x Vasco)
# id_evento, id_partida, id_jogador, tipo_evento, minuto_jogo, revisado_var
data = [
    (1, 101, 10, "Gol", 15, False),             # Gol do camisa 10
    (2, 101, 5, "Cartão Amarelo", 30, False),    # Falta do volante
    (3, 101, 9, "Gol", 44, False)              # Gol do centroavante
]

columns = ["id_evento", "id_partida", "id_jogador", "tipo_evento", "minuto_jogo", "revisado_var"]

# 3. Criando o DataFrame do Spark
df_inicial = spark.createDataFrame(data, columns)

# 4. Salvando no formato Delta (Operação: INSERT inicial)
df_inicial.write.format("delta").mode("overwrite").save(path_delta)

# 5. Lendo e exibindo a tabela para conferir
df_check = spark.read.format("delta").load(path_delta)
df_check.show()

26/05/04 23:32:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        2|       101|         5|Cartão Amarelo|         30|       false|
|        3|       101|         9|           Gol|         44|       false|
|        1|       101|        10|           Gol|         15|       false|
+---------+----------+----------+--------------+-----------+------------+



In [3]:
from delta.tables import *

# 1. Criar o objeto DeltaTable (essencial para comandos de atualização e exclusão)
delta_table = DeltaTable.forPath(spark, path_delta)

# 2. Cenário VAR 1: Corrigindo o autor do gol (id_evento = 1)
print(">>> VAR analisando o primeiro gol...")
delta_table.update(
    condition = "id_evento = 1",
    set = { "id_jogador": "99", "revisado_var": "True" }
)

# 3. Cenário VAR 2: Anulando o gol por impedimento (id_evento = 3)
print(">>> VAR anulando o segundo gol por impedimento...")
delta_table.delete("id_evento = 3")

# 4. Exibir a tabela final atualizada
print("\nPlacar Final atualizado pelo VAR:")
delta_table.toDF().orderBy("id_evento").show()

>>> VAR analisando o primeiro gol...


26/05/04 23:33:51 WARN UpdateCommand: Could not validate number of records due to missing statistics.
                                                                                

>>> VAR anulando o segundo gol por impedimento...


26/05/04 23:33:53 WARN DeleteCommand: Could not validate number of records due to missing statistics.



Placar Final atualizado pelo VAR:
+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        99|           Gol|         15|        true|
|        2|       101|         5|Cartão Amarelo|         30|       false|
+---------+----------+----------+--------------+-----------+------------+



In [4]:
# Viagem no Tempo: Consultando os dados originais (antes do VAR)
print(">>> Consultando a Versão 0 (Antes da intervenção do VAR):")
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(path_delta)
df_v0.orderBy("id_evento").show()

>>> Consultando a Versão 0 (Antes da intervenção do VAR):
+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        10|           Gol|         15|       false|
|        2|       101|         5|Cartão Amarelo|         30|       false|
|        3|       101|         9|           Gol|         44|       false|
+---------+----------+----------+--------------+-----------+------------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51742)
Traceback (most recent call last):
  File "/home/arthurlg81/.pyenv/versions/3.12.1/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/arthurlg81/.pyenv/versions/3.12.1/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/home/arthurlg81/.pyenv/versions/3.12.1/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/arthurlg81/.pyenv/versions/3.12.1/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/home/arthurlg81/trabalho-spark/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/home/arthurlg81/trabalho-spark/.venv/lib/python3.12/site-packages/pyspark/accum